# MODIS Aqua **particulate backscatter bbp(443)** over DBO — streaming, EASE-Grid 2.0 North

bbp companion to the chl / SST / cloud / sea-ice notebooks. Streams the **MODIS-Aqua Level-3 monthly
mapped particulate backscattering coefficient at 443 nm** (`MODISA_L3m_IOP`, GIOP suite, 4 km),
grids each month onto the **same EASE-2 4 km DBO grid**, and folds it into **in-memory accumulators**
(nothing written but a checkpoint). Same through-time analysis as the others: per-box monthly +
deseasonalized anomaly + Theil-Sen/AR(1) Mann-Kendall, seasonal calendar, anomaly GIF, per-pixel trend map.

### Product
- **`MODISA_L3m_IOP`** v2022.0 (concept_id `C3380709177-OB_CLOUD`), file pattern
  `*.MO.IOP.bbp_443.4km.nc` -> **281 monthly granules, 2002-07 -> 2025-11**.
- Variable **`bbp_443`** (units m^-1) = particulate backscattering at 443 nm, a proxy for particle
  (often phytoplankton + detritus) load. Global 4 km lat/lon grid; each file is one month.
- bbp is a positive, roughly log-distributed quantity. Here we keep **linear** bbp with deseasonalized
  monthly anomalies (consistent with the other DBO notebooks); a log10 variant is noted in S8.

### Reach
- Ocean color needs **open water + sunlight**, so all DBO boxes get data in their open-water season;
  the high-latitude boxes (DBO3-8) are **summer-only** (polar night + ice in winter -> NaN months).

## 1. Imports

In [1]:
import time as _time, warnings, gc
from pathlib import Path
from datetime import date

import numpy as np, pandas as pd, xarray as xr
import earthaccess
from pyresample import AreaDefinition, geometry
from pyresample.kd_tree import resample_nearest
from matplotlib.path import Path as MplPath
import matplotlib.pyplot as plt, matplotlib.dates as mdates
import cartopy.crs as ccrs, cartopy.feature as cfeature
from concurrent.futures import ThreadPoolExecutor, wait, FIRST_COMPLETED
from scipy.stats import theilslopes, kendalltau, norm
warnings.filterwarnings("ignore")
print("imports OK")

imports OK


##2. Configuration - product, EASE-2 grid, DBO boxes

In [2]:
# >>> confirm with S3 (search) and S3b (inspect) <<<
BBP_SHORT    = "MODISA_L3m_IOP"
BBP_CONCEPT  = "C3380709177-OB_CLOUD"            # MODISA L3m IOP (GIOP), v2022.0
GRANULE_PAT  = "*.MO.IOP.bbp_443.4km.nc"         # monthly, bbp(443), 4 km -> one file per month
BBP_VAR      = "bbp_443"                          # particulate backscatter at 443 nm (m^-1)
LATN, LONN   = "lat", "lon"

START_DATE = date(2002, 7, 1)                     # MODIS-Aqua ocean-color record start
END_DATE   = date.today()
LAT_MIN, LAT_MAX = 45, 90
ROI = 6_000                                       # nearest-neighbour radius (m); L3m is 4 km
N_WORKERS = 8

# DBO bounding box (lon0, lat0, lon1, lat1) - coarse pre-subset before regridding
DBO_BBOX = (-180.0, 50.0, -124.0, 74.0)

dbo_area = AreaDefinition(
    area_id="ease2_north_dbo_4km",
    description="EASE-Grid 2.0 North subset: Aleutians -> Beaufort (DBO1-8), 4 km",
    proj_id="EPSG:6931",
    projection={"proj":"laea","lat_0":90,"lon_0":0,"x_0":0,"y_0":0,"datum":"WGS84","units":"m"},
    width=906, height=845, area_extent=(-3_624_000, 992_000, 0, 4_372_000))

CHECKPOINT = True
CKPT_PATH  = Path.home() / "modis_bbp_dbo_ckpt.pkl"  # add your path
CKPT_EVERY = 60
CACHE_DERIVED = False
CACHE_PATH    = Path.home() / "modis_bbp_dbo_products.pkl"  # add your path

DBO_BOXES = {
    "DBO1": [(63.769462,-172.312422),(61.945988,-172.186927),(61.846516,-175.790978),(63.662718,-176.147111)],
    "DBO2": [(65.107747,-170.865176),(65.097001,-167.859549),(64.481828,-167.907758),(64.493658,-170.840669)],
    "DBO3": [(66.785880,-171.330008),(68.608869,-171.419334),(68.572412,-166.481029),(66.752281,-166.755808)],
    "DBO4": [(71.994000,-164.226000),(71.801000,-159.729000),(70.641000,-160.305000),(70.821000,-164.556000)],
    "DBO5": [(71.807730,-158.537691),(71.635583,-155.931220),(71.110941,-156.307277),(71.278277,-158.847920)],
    "DBO6": [(71.850000,-150.108000),(70.890000,-150.976000),(71.159000,-153.865000),(72.119000,-152.980000)],
    "DBO7": [(70.868000,-141.528000),(69.946000,-142.679000),(70.333000,-145.439000),(71.253000,-144.256000)],
    "DBO8": [(70.834000,-125.155000),(70.049000,-126.997000),(70.664000,-129.354000),(71.442000,-127.431000)],
}
data_crs = ccrs.LambertAzimuthalEqualArea(central_latitude=90, central_longitude=0)
disp     = ccrs.LambertAzimuthalEqualArea(central_latitude=90, central_longitude=-152)
x0, y0, x1, y1 = dbo_area.area_extent
try: import cmocean; BBP_CMAP = cmocean.cm.turbid
except ImportError: BBP_CMAP = "viridis"
print(f"{BBP_SHORT} {GRANULE_PAT} | {START_DATE} -> {END_DATE} | grid {dbo_area.shape} | streaming (no disk)")

MODISA_L3m_IOP *.MO.IOP.bbp_443.4km.nc | 2002-07-01 -> 2026-06-29 | grid (845, 906) | streaming (no disk)


## 3. Auth + search the monthly granules

In [3]:
auth = earthaccess.login(persist=True)

def _search(s, e, limit=2000):
    for k in range(5):
        try:
            return earthaccess.search_data(concept_id=BBP_CONCEPT, granule_name=GRANULE_PAT,
                                           temporal=(s, e), count=limit)
        except Exception as ex:
            print("  retry:", str(ex)[:80]); _time.sleep(2*(k+1))
    raise RuntimeError("CMR search failed")

def granule_date(g):
    # AQUA_MODIS.YYYYMMDD_YYYYMMDD.L3m.MO.IOP.bbp_443.4km.nc -> month start date
    d = g.data_links()[0].rsplit("/", 1)[-1].split(".")[1].split("_")[0]
    return date(int(d[:4]), int(d[4:6]), 1)

all_granules = _search(START_DATE.isoformat(), END_DATE.isoformat())
granules_by_month = {}
for g in all_granules:
    granules_by_month.setdefault(granule_date(g), g)
print(f"{len(all_granules)} monthly granules | {len(granules_by_month)} unique months "
      f"({min(granules_by_month)} -> {max(granules_by_month)})" if granules_by_month else "no granules")
if all_granules:
    print("example:", all_granules[0].data_links()[0].rsplit('/',1)[-1])

281 monthly granules | 281 unique months (2002-07-01 -> 2025-11-01)
example: AQUA_MODIS.20020701_20020731.L3m.MO.IOP.bbp_443.4km.nc


## 4. Reader - stream one month -> subset DBO bbox -> regrid onto EASE-2

In [5]:
def _read_bbp(f):
    """L3m global grid -> DBO-bbox subset (lat 1-D, lon 1-D, bbp 2-D, positive only)."""
    with xr.open_dataset(f, engine="h5netcdf") as ds:
        lat = np.asarray(ds[LATN].values, float); lon = np.asarray(ds[LONN].values, float)
        lon0, la0, lon1, la1 = DBO_BBOX
        jla = np.where((lat >= la0) & (lat <= la1))[0]
        ilo = np.where((lon >= lon0) & (lon <= lon1))[0]
        if jla.size == 0 or ilo.size == 0: return None
        sub = ds[BBP_VAR].isel({LATN: slice(int(jla[0]), int(jla[-1]) + 1),
                                LONN: slice(int(ilo[0]), int(ilo[-1]) + 1)})
        v  = np.asarray(sub.values, float).squeeze()
        la = np.asarray(sub[LATN].values, float); lo = np.asarray(sub[LONN].values, float)
    v = np.where(v > 0, v, np.nan)          # bbp is strictly positive; drop fills / non-positive
    return la, lo, v

def grid_to_area(lat, lon, val, area):
    m = np.isfinite(val)
    swath = geometry.SwathDefinition(lons=lon[m], lats=lat[m])
    return resample_nearest(swath, val[m], area, radius_of_influence=ROI,
                            fill_value=np.nan, reduce_data=False).astype(np.float32)

def project_month(granule, area):
    f = earthaccess.open([granule], show_progress=False)[0]
    try: out = _read_bbp(f)
    finally:
        try: f.close()
        except Exception: pass
    if out is None: return np.full(area.shape, np.nan, np.float32)
    la, lo, v = out
    lon2d, lat2d = np.meshgrid(lo, la)
    return grid_to_area(lat2d, lon2d, v, area)

## 5. DBO box masks on the EASE-2 grid

In [6]:
lons2d, lats2d = dbo_area.get_lonlats()
grid_pts = np.column_stack([lons2d.ravel(), lats2d.ravel()])
box_masks = {name: MplPath([(lo,la) for la,lo in corners] + [(corners[0][1], corners[0][0])])
                       .contains_points(grid_pts).reshape(lons2d.shape)
             for name, corners in DBO_BOXES.items()}
for name, m in box_masks.items():
    print(f"  {name}: {int(m.sum()):>6,} grid cells")

  DBO1:  2,412 grid cells
  DBO2:    605 grid cells
  DBO3:  2,561 grid cells
  DBO4:  1,293 grid cells
  DBO5:    351 grid cells
  DBO6:    739 grid cells
  DBO7:    773 grid cells
  DBO8:    760 grid cells


## 6. Streaming pass (parallel, checkpointed) - build everything

Builds `box_ts` (monthly bbp per box), `mean_map` (record-mean), and `annual_bbp` (per-year mean).
Checkpoints every `CKPT_EVERY` months so a crash resumes.

In [7]:
import pickle
def _have_products():
    return all(k in globals() and globals()[k] is not None for k in ("box_ts","mean_map","annual_bbp"))
ny, nx = dbo_area.shape

if CACHE_DERIVED and CACHE_PATH.exists() and not _have_products():
    _d = pickle.load(open(CACHE_PATH, "rb"))
    box_ts, mean_map, annual_bbp = _d["box_ts"], _d["mean_map"], _d["annual_bbp"]
    print("loaded finished products - no streaming.")

if _have_products():
    print("products already in memory - skipping the stream.")
else:
    if CHECKPOINT and CKPT_PATH.exists():
        _ck = pickle.load(open(CKPT_PATH, "rb"))
        mean_sum, mean_cnt, ann, box_rows = _ck["mean_sum"], _ck["mean_cnt"], _ck["ann"], _ck["box_rows"]
        print(f"resumed: {len(box_rows)} months already folded.")
    else:
        mean_sum = np.zeros((ny, nx), np.float32); mean_cnt = np.zeros((ny, nx), np.float32)
        ann = {}; box_rows = {}

    def _stream_one(item):
        d, g = item
        try: return d, project_month(g, dbo_area), None
        except Exception as e: return d, None, f"{type(e).__name__}: {str(e)[:70]}"

    def _fold(d, arr):
        m = np.isfinite(arr)
        mean_sum[m] += arr[m]; mean_cnt[m] += 1
        s = ann.setdefault(d.year, [np.zeros((ny, nx), np.float32), np.zeros((ny, nx), np.float32)])
        s[0][m] += arr[m]; s[1][m] += 1
        bm = {}
        for n, mask in box_masks.items():
            v = arr[mask & m]; bm[n] = float(np.mean(v)) if v.size else np.nan
        box_rows[d] = bm

    def _save_ckpt():
        if not CHECKPOINT: return
        CKPT_PATH.parent.mkdir(parents=True, exist_ok=True)
        tmp = CKPT_PATH.with_suffix(".tmp")
        pickle.dump({"mean_sum": mean_sum, "mean_cnt": mean_cnt, "ann": ann, "box_rows": box_rows}, open(tmp, "wb"))
        tmp.replace(CKPT_PATH)

    remaining = [(d, g) for d, g in sorted(granules_by_month.items()) if d not in box_rows]
    src = iter(remaining); MAXQ = max(2, N_WORKERS*2); t0 = _time.time(); n_ok = n_fail = since = 0
    print(f"{len(box_rows)} done | streaming {len(remaining)} more...")
    with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
        inflight = set()
        for _ in range(MAXQ):
            try: inflight.add(ex.submit(_stream_one, next(src)))
            except StopIteration: break
        while inflight:
            done, inflight = wait(inflight, return_when=FIRST_COMPLETED)
            for fut in done:
                d, arr, err = fut.result()
                if arr is None:
                    n_fail += 1
                    if n_fail <= 10: print(f"  {d}  X {err}")
                else:
                    _fold(d, arr); n_ok += 1; since += 1
                    if n_ok % 50 == 0: print(f"  {n_ok} ok / {n_fail} failed  ({(_time.time()-t0)/60:.1f} min)")
                    if since >= CKPT_EVERY: _save_ckpt(); since = 0
                try: inflight.add(ex.submit(_stream_one, next(src)))
                except StopIteration: pass
            del done
    _save_ckpt()

    dates = sorted(box_rows); tcoord = np.array([np.datetime64(d.isoformat()) for d in dates])
    box_ts = {n: xr.DataArray(np.array([box_rows[d][n] for d in dates]), dims="time", coords={"time": tcoord})
              for n in box_masks}
    mean_map = np.where(mean_cnt > 0, mean_sum / mean_cnt, np.nan)
    _years = sorted(ann)
    annual_bbp = xr.DataArray(np.stack([np.where(ann[y][1] > 0, ann[y][0]/ann[y][1], np.nan) for y in _years]),
                              dims=("year","y","x"), coords={"year": _years})
    gc.collect()
    print(f"\nfolded {len(box_rows)} months -> box_ts, mean_map, {len(_years)} annual frames.")

if CACHE_DERIVED and not CACHE_PATH.exists():
    pickle.dump({"box_ts": box_ts, "mean_map": mean_map, "annual_bbp": annual_bbp}, open(CACHE_PATH, "wb"))
    print("saved finished products.")

resumed: 257 months already folded.
257 done | streaming 24 more...
  2004-12-01  X ValueError: Cannot resample empty data set
  2005-12-01  X ValueError: Cannot resample empty data set
  2008-12-01  X ValueError: Cannot resample empty data set
  2007-12-01  X ValueError: Cannot resample empty data set
  2002-12-01  X ValueError: Cannot resample empty data set
  2003-12-01  X ValueError: Cannot resample empty data set
  2006-12-01  X ValueError: Cannot resample empty data set
  2011-12-01  X ValueError: Cannot resample empty data set
  2010-12-01  X ValueError: Cannot resample empty data set
  2009-12-01  X ValueError: Cannot resample empty data set

folded 257 months -> box_ts, mean_map, 24 annual frames.


## 7. Trend helpers

In [8]:
def _lag1(x):
    x = np.asarray(x, float); a, b = x[1:], x[:-1]; v = np.isfinite(a) & np.isfinite(b)
    return float(np.corrcoef(a[v], b[v])[0, 1]) if v.sum() >= 5 else 0.0
def mk(x, y, minn=24):
    v = np.isfinite(x) & np.isfinite(y); x, y = x[v], y[v]; n = len(x)
    if n < minn: return dict(slope=np.nan, intercept=np.nan, p=np.nan, p_naive=np.nan, n=n)
    slope, intercept, _, _ = theilslopes(y, x)
    tau, p_naive = kendalltau(x, y)
    sl, ic = np.polyfit(x, y, 1); phi = _lag1(y - (sl*x + ic))
    ne = n*(1-phi)/(1+phi) if (abs(phi) < 1 and phi != 0) else n
    p = 2*(1 - norm.cdf(abs(tau/np.sqrt(2*(2*ne+5)/(9*ne*(ne-1)))))) if ne > 4 else np.nan
    return dict(slope=slope, intercept=intercept, p=p, p_naive=p_naive, n=n)
def _stars(p):
    return ("***" if p<0.001 else "**" if p<0.01 else "*" if p<0.05 else "ns") if np.isfinite(p) else "n/a"
print("helpers ready")

helpers ready


## 8. Per-DBO-box monthly bbp + month-of-year anomaly trend (Theil-Sen + AR(1) MK)

bbp is already monthly, so we deseasonalize by month-of-year and trend the anomaly. Trend is reported
as % of the box mean per decade (bbp magnitudes differ a lot between boxes).
*log10 variant:* replace `s` with `np.log10(s)` before the anomaly step to trend relative change.

In [9]:
names = [n for n in DBO_BOXES if n in box_ts and np.isfinite(box_ts[n]).sum() > 24]
fig, axes = plt.subplots(2, 4, figsize=(20, 8), sharex=True); axes = axes.ravel()
for ax, name in zip(axes, names):
    s = box_ts[name]
    moy = s["time"].dt.month
    clim = s.assign_coords(moy=("time", moy.values)).groupby("moy").mean()
    anom = s.assign_coords(moy=("time", moy.values)).groupby("moy") - clim
    t = s["time"].values; dec = pd.to_datetime(t); dec = np.asarray(dec.year + (dec.month-0.5)/12.0, float)
    r = mk(dec, anom.values); base = float(np.nanmean(s.values))
    ax.plot(t, s.values, "-", color="#9bbf9b", lw=0.5, alpha=0.6)
    ax.plot(t, anom.values, "-", color="#1f7a1f", lw=0.7)
    ax.axhline(0, color="gray", lw=0.5)
    if np.isfinite(r["slope"]):
        ax.plot(t, r["intercept"] + r["slope"]*dec, "-", color="#d6602c", lw=1.6)
        pct = 100*r["slope"]*10/base if base else np.nan
        lab = f"{r['slope']*10:+.1e}/dec ({pct:+.1f}%/dec) {_stars(r['p'])}"
    else: lab = f"(n={r['n']}, insufficient)"
    ax.set_title(f"{name}  {lab}", fontsize=8.5, loc="left"); ax.grid(alpha=0.25)
    ax.xaxis.set_major_locator(mdates.YearLocator(base=5)); ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    for sp in ("top","right"): ax.spines[sp].set_visible(False)
for ax in axes[len(names):]: ax.axis("off")
for ax in axes[::4]: ax.set_ylabel("bbp(443) (m$^{-1}$)")
fig.suptitle("MODIS-Aqua bbp(443) per DBO box - monthly (light) + deseasonalized anomaly + Theil-Sen/AR(1) MK trend",
             fontsize=13, y=1.0)
plt.tight_layout(); plt.show()

## 9. Per-box bbp trend table (deseasonalized monthly anomaly, MK naive & AR(1))

In [10]:
rows = []
for name in names:
    s = box_ts[name]; moy = s["time"].dt.month
    anom = s.assign_coords(moy=("time", moy.values)).groupby("moy") - \
           s.assign_coords(moy=("time", moy.values)).groupby("moy").mean()
    t = pd.to_datetime(s["time"].values); dec = np.asarray(t.year + (t.month-0.5)/12.0, float)
    r = mk(dec, anom.values); base = float(np.nanmean(s.values))
    rows.append((name, base, r["slope"]*10, 100*r["slope"]*10/base if base else np.nan,
                 r["p_naive"], r["p"], _stars(r["p"]), r["n"]))
tbl = pd.DataFrame(rows, columns=["box","mean_bbp","trend(/dec)","trend(%/dec)","p_naive","p_AR1","sig","n_months"]).set_index("box")
pd.set_option("display.float_format", lambda v: f"{v:.4g}")
print("MODIS bbp(443) trend per DBO box (deseasonalized monthly, Theil-Sen + MK):\n"); print(tbl)

MODIS bbp(443) trend per DBO box (deseasonalized monthly, Theil-Sen + MK):

      mean_bbp  trend(/dec)  trend(%/dec)  p_naive   p_AR1 sig  n_months
box                                                                     
DBO1   0.00452    0.0003686         8.156 0.002171 0.02039   *       178
DBO2  0.006699    5.894e-05        0.8798   0.7474  0.8349  ns       170
DBO3  0.007388    0.0001873         2.535   0.3304  0.2627  ns       150
DBO4  0.006272    2.905e-05        0.4632   0.8835  0.8907  ns       129
DBO5  0.008934   -0.0003498        -3.915   0.3469  0.4205  ns       136
DBO6  0.008683   -0.0001576        -1.815   0.7232  0.7251  ns       111
DBO7   0.00824     0.000282         3.422   0.5629  0.6311  ns       111
DBO8  0.008531    0.0006583         7.717  0.04855  0.1349  ns       144


In [11]:
# === Stream MODIS-Aqua monthly chlorophyll (chlor_a) onto the SAME boxes as bbp ==========
import pickle
CHL_CONCEPT = "C3380709133-OB_CLOUD"; CHL_PAT = "*.MO.CHL.chlor_a.4km.nc"; CHL_VAR = "chlor_a"
CHL_CKPT = Path.home() / "modis_chl_dbo_ckpt.pkl"  # add your path

def _chl_date(g):
    d = g.data_links()[0].rsplit("/",1)[-1].split(".")[1].split("_")[0]; return date(int(d[:4]), int(d[4:6]), 1)
def project_chl(g, area):
    f = earthaccess.open([g], show_progress=False)[0]
    try:
        with xr.open_dataset(f, engine="h5netcdf") as ds:
            lat=np.asarray(ds["lat"].values,float); lon=np.asarray(ds["lon"].values,float); lo0,la0,lo1,la1=DBO_BBOX
            jla=np.where((lat>=la0)&(lat<=la1))[0]; ilo=np.where((lon>=lo0)&(lon<=lo1))[0]
            sub=ds[CHL_VAR].isel(lat=slice(int(jla[0]),int(jla[-1])+1), lon=slice(int(ilo[0]),int(ilo[-1])+1))
            v=np.asarray(sub.values,float).squeeze(); la=np.asarray(sub["lat"].values,float); lo=np.asarray(sub["lon"].values,float)
    finally:
        try: f.close()
        except Exception: pass
    v=np.where(v>0,v,np.nan)
    if not np.isfinite(v).any(): return np.full(area.shape, np.nan, np.float32)   # dark/empty month
    lon2d,lat2d=np.meshgrid(lo,la); return grid_to_area(lat2d, lon2d, v, area)

chl_rows = pickle.load(open(CHL_CKPT,"rb")) if CHL_CKPT.exists() else {}
gch = earthaccess.search_data(concept_id=CHL_CONCEPT, granule_name=CHL_PAT,
                              temporal=(START_DATE.isoformat(), END_DATE.isoformat()), count=2000)
chl_by_month = {}
for g in gch: chl_by_month.setdefault(_chl_date(g), g)
todo = [(d,g) for d,g in sorted(chl_by_month.items()) if d not in chl_rows]
print(f"{len(chl_rows)} done | streaming {len(todo)} chl months...")
def _one(it):
    d,g=it
    try:
        arr=project_chl(g, dbo_area)
        return d, {n: (float(np.nanmean(arr[m & np.isfinite(arr)])) if (m & np.isfinite(arr)).any() else np.nan)
                   for n,m in box_masks.items()}, None
    except Exception as e: return d, None, f"{type(e).__name__}: {str(e)[:50]}"
nok=nf=0
with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
    for d,bm,err in ex.map(_one, todo):
        if bm is None: nf+=1; continue
        chl_rows[d]=bm; nok+=1
        if nok%50==0: print(f"  {nok} ok / {nf} failed")
pickle.dump(chl_rows, open(CHL_CKPT,"wb"))
dts=sorted(chl_rows); tc=np.array([np.datetime64(d.isoformat()) for d in dts])
chl_box_ts={n: xr.DataArray(np.array([chl_rows[d][n] for d in dts]), dims="time", coords={"time":tc}) for n in DBO_BOXES}
print(f"chl_box_ts ready: {len(dts)} months | saved to {CHL_CKPT.name}")


281 done | streaming 0 chl months...
chl_box_ts ready: 281 months | saved to modis_chl_dbo_ckpt.pkl
